<a href="https://colab.research.google.com/github/Vigneshkumarcvk/NLP-Model/blob/main/NLP_Practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

In [13]:
df = pd.read_csv("IMDB Dataset.csv")

In [14]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [16]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [17]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [23]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
  text = re.sub('[^a-zA-Z]', ' ', text)
  text = text.lower()
  text = text.split()
  text = [ps.stem(word) for word in text if word not in stop_words]
  return ''.join(text)

In [26]:
df['clean_review'] = df['review'].apply(clean_text)

In [29]:
x = df['clean_review']
y = df['sentiment']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state = 42)

In [41]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2)
)

x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)


In [43]:
model = LogisticRegression(max_iter = 1000)
model.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [44]:
y_pred = model.predict(x_test_tfidf)
print("Accuracy:",accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusionMatrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.4971

Classification Report:
               precision    recall  f1-score   support

           0       0.50      1.00      0.66      4961
           1       1.00      0.00      0.00      5039

    accuracy                           0.50     10000
   macro avg       0.75      0.50      0.33     10000
weighted avg       0.75      0.50      0.33     10000


ConfusionMatrix:
 [[4961    0]
 [5029   10]]


In [45]:
def predict_sentiment(review):
  review = clean_text(review)
  review_tfidf = tfidf.transform([review])
  prediction = model.predict(review_tfidf)
  return"Positive" if prediction[0] == 1 else "Negative"

  print(predict_sentiment("The movie was fantastic and very emotional"))
  predict_sentiment("Worst movie I have never seen")

In [48]:
test_reviews = [
    "I absolutely loved the movie, brilliant acting",
    "The film was boring and too long",
    "Amazing story and great direction",
    "Worst movie, waste of time",
    "It was okay, not bad but not great",
    "The movie was fantastic and very emotional"
]

for review in test_reviews:
    print(review, "➡️", predict_sentiment(review))


I absolutely loved the movie, brilliant acting ➡️ Negative
The film was boring and too long ➡️ Negative
Amazing story and great direction ➡️ Negative
Worst movie, waste of time ➡️ Negative
It was okay, not bad but not great ➡️ Negative
The movie was fantastic and very emotional ➡️ Negative
